In [1]:
import torch
import torch.nn as nn
import os
import onnx
import onnxruntime as ort
import numpy as np

# ================= 配置区域 =================
# 模型权重路径 (根据你的 main_train_tensor.py 设置)
MODEL_PATH = "./training_results_tensor/best_model.pth"
# 输出 ONNX 文件路径
ONNX_PATH = "./training_results_tensor/best_model.onnx"

# 类别定义 (用于确定输出维度)
CLASS_NAMES = [
    "BPSK", "QPSK", "8PSK", "OQPSK", "PI4DQPSK", 
    "16QAM", "32QAM", "64QAM", 
    "2ASK", "16APSK", "32APSK", "FSK2"
]
NUM_CLASSES = len(CLASS_NAMES)

# ================= 模型定义 =================
# 注意：必须与 main_train_tensor.py 中的定义完全一致
class TinyCNN(nn.Module):
    def __init__(self, num_classes):
        super(TinyCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Dropout2d(0.1),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Dropout2d(0.1),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*8*8, 128),
            nn.ReLU(), 
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )
    def forward(self, x): return self.classifier(self.features(x))

# ================= 导出逻辑 =================
def export_model():
    print(f"Loading model from {MODEL_PATH}...")
    
    # 1. 实例化模型并加载权重
    # map_location='cpu' 确保即使是在 GPU 上训练的模型也能在仅有 CPU 的机器上导出
    device = torch.device("cpu")
    model = TinyCNN(NUM_CLASSES).to(device)
    
    try:
        state_dict = torch.load(MODEL_PATH, map_location=device)
        model.load_state_dict(state_dict)
    except FileNotFoundError:
        print(f"错误：找不到模型文件 {MODEL_PATH}，请先运行 main_train_tensor.py 进行训练。")
        return

    # 2. 切换到评估模式 (非常重要！)
    # 这会固定 BatchNorm 的均值/方差，并禁用 Dropout
    model.eval()

    # 3. 创建虚拟输入 (Dummy Input)
    # 形状必须是 (Batch_Size, Channels, Height, Width)
    # 这里 Batch_Size 设为 1，Channels=3, Size=64
    dummy_input = torch.randn(1, 3, 64, 64, device=device)

    print(f"Exporting to {ONNX_PATH}...")

    # 4. 执行导出
    torch.onnx.export(
        model,                      # 模型实例
        dummy_input,                # 虚拟输入
        ONNX_PATH,                  # 输出路径
        export_params=True,         # 是否保存权重
        opset_version=11,           # ONNX 版本 (11 是最通用的)
        do_constant_folding=True,   # 是否执行常量折叠优化
        input_names=['input'],      # 输入节点名称 (C++中会用到)
        output_names=['output'],    # 输出节点名称 (C++中会用到)
        dynamic_axes={              # 动态轴设置 (允许可变 Batch Size)
            'input': {0: 'batch_size'},
            'output': {0: 'batch_size'}
        }
    )
    print("Export successful!")

# ================= 验证逻辑 =================
def verify_onnx():
    print("\nVerifying ONNX model...")
    
    # 1. 检查模型结构是否完整
    onnx_model = onnx.load(ONNX_PATH)
    onnx.checker.check_model(onnx_model)
    print("ONNX structure check passed.")

    # 2. 使用 ONNX Runtime 进行一次推理测试
    # 创建推理会话
    sess = ort.InferenceSession(ONNX_PATH)
    
    # 获取输入输出名称
    input_name = sess.get_inputs()[0].name
    output_name = sess.get_outputs()[0].name
    print(f"Model Input Name: {input_name}")   # 应该是 'input'
    print(f"Model Output Name: {output_name}") # 应该是 'output'
    
    # 构造随机数据进行测试
    x = np.random.randn(1, 3, 64, 64).astype(np.float32)
    
    # 运行推理
    outputs = sess.run([output_name], {input_name: x})
    
    print(f"Inference test passed. Output shape: {outputs[0].shape}")
    print(f"Model is ready for C++ deployment at: {os.path.abspath(ONNX_PATH)}")

if __name__ == "__main__":
    export_model()
    if os.path.exists(ONNX_PATH):
        verify_onnx()


Loading model from ./training_results_tensor/best_model.pth...
Exporting to ./training_results_tensor/best_model.onnx...
Export successful!

Verifying ONNX model...
ONNX structure check passed.
Model Input Name: input
Model Output Name: output
Inference test passed. Output shape: (1, 12)
Model is ready for C++ deployment at: e:\Project\AMR-Benchmark\Other\training_results_tensor\best_model.onnx
